# gan-image-gen — Quickstart Notebook

This notebook mirrors the CLI workflow in the `.py` scripts, but in an interactive format.

Real images are sourced from the **Fruits 360** dataset (subset: `apple`, `banana`, `orange`). The training code expects a prepared folder layout:

```
data_final/
  train/<class>/*
  val/<class>/*
  test/<class>/*
```

Notes:
- By default, this repo ignores `data_final/` and `runs/` in git for public publishing.
- FID uses an InceptionV3 backbone; torchvision may download weights the first time.
- To run fully offline, set `fid_every=0` (disables FID).


In [ ]:
from pathlib import Path

import torch

from config import Config, resolve_device

cfg = Config()
device = torch.device(resolve_device(cfg.device))

print("torch:", torch.__version__)
print("device:", device)
print("data_root:", cfg.data_root.resolve())


In [ ]:
from torchvision import datasets, transforms

data_root = Path(cfg.data_root)
train_dir = data_root / "train"
val_dir = data_root / "val"
test_dir = data_root / "test"

for p in [train_dir, val_dir, test_dir]:
    print(f"{p} exists: {p.exists()}")

if not train_dir.exists():
    raise FileNotFoundError(
        "Missing dataset folder. Create data_final/{train,val,test}/<class>/... first."
    )

tf = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.ToTensor(),
])
train_ds = datasets.ImageFolder(str(train_dir), transform=tf)

print("classes:", train_ds.classes)
print("train size:", len(train_ds))


In [ ]:
import matplotlib.pyplot as plt
import torch
from torchvision.utils import make_grid

imgs = torch.stack([train_ds[i][0] for i in range(min(32, len(train_ds)))])
grid = make_grid(imgs, nrow=8)

plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0))
plt.axis("off")
plt.show()


## Optional: GAN smoke test (very small)

This runs for **1 epoch** and disables FID to avoid downloading Inception weights.
Artifacts are written under `runs/gan/`.

In [ ]:
from train_gan import train

quick_cfg = Config(
    gan_epochs=1,
    gan_batch=16,
    num_workers=0,
    persistent_workers=False,
    fid_every=0,
    ckpt_every=1,
    sample_every=1,
)

G, D = train(quick_cfg)
gan_device = next(G.parameters()).device
print("GAN device:", gan_device)


### Visualize a generated batch (not saved)

In [ ]:
import torch
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

G.eval()
z = torch.randn(24, quick_cfg.z_dim, device=gan_device)
y = torch.tensor([0] * 8 + [1] * 8 + [2] * 8, device=gan_device)

with torch.no_grad():
    fake = G(z, y).cpu()

grid = make_grid(fake, nrow=8, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(10, 4))
plt.imshow(grid.permute(1, 2, 0))
plt.axis("off")
plt.show()


## Optional: classifier smoke test

Trains for **1 epoch** on a tiny subset to verify the pipeline.

In [ ]:
from train_classifier import run

clf_cfg = Config(
    clf_epochs=1,
    clf_batch=16,
    num_workers=0,
    persistent_workers=False,
)

result = run(clf_cfg, scenario="real", n_per_class=20, synth_dir="data_synth", out_dir="runs/clf")
result
